# COVID-19 Mortality Prediction
## Data Preparation for Explainable AI and Clustering

### Overview

This notebook begins the next phase of the COVID-19 Mortality Prediction project by preparing the data for several new analytical components that build upon the machine learning work completed in Capstone Two.

The original project developed and evaluated multiple machine learning models for predicting country-level COVID-19 mortality using demographic, healthcare, economic, vaccination, and government policy variables. The Random Forest model achieved the strongest predictive performance and serves as the foundation for this capstone.

Rather than repeating the original data acquisition, cleaning, and preprocessing steps, this notebook begins with the cleaned analytical dataset produced during Capstone Two and performs the additional data preparation required for the new components introduced in Capstone Three.

---

## Objectives

The objectives of this notebook are to:

- Load and validate the cleaned analytical dataset from Capstone Two.
- Verify compatibility between the dataset and the trained Random Forest model.
- Prepare standardized datasets for country clustering analysis.
- Create reusable data assets that support explainable AI (SHAP) and future dashboard enhancements.
- Save all intermediate datasets required for subsequent exploratory analysis and modeling.

---

## Relationship to Capstone Two

The following components were completed during Capstone Two and will be reused throughout this project:

- Data acquisition and integration
- Data cleaning and quality assessment
- Feature engineering
- Model preprocessing
- Random Forest model development and evaluation
- Interactive Streamlit dashboard

Capstone Three extends this work by introducing:

- Explainable AI using SHAP (SHapley Additive exPlanations)
- Country clustering using unsupervised learning
- Enhanced decision-support capabilities within the Streamlit dashboard
- *(Optional)* AI-assisted research assistant for natural language interaction

---

## Expected Outputs

At the completion of this notebook, the project will contain:

- A validated analytical dataset
- Standardized feature matrix for clustering
- Reusable datasets for downstream analysis
- Saved preprocessing assets for subsequent notebooks

These outputs will serve as the foundation for the exploratory analysis and modeling notebooks that follow.

In [1]:
packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "sklearn",
    "scipy",
    "statsmodels"
]

for pkg in packages:
    try:
        __import__(pkg)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"✗ {pkg}")

✓ numpy
✓ pandas
✓ matplotlib
✓ seaborn
✓ sklearn
✓ scipy
✓ statsmodels


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from pathlib import Path
import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


In [3]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

assert (PROJECT_ROOT / "data").exists(), (
    "Project root not found."
)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
MODEL_DIR = PROJECT_ROOT / "models"

print(f"Project: {PROJECT_ROOT.name}")

# Source Project (Capstone 2)
SOURCE_PROJECT = PROJECT_ROOT.parent / "Capstone2"

SOURCE_MODEL_DIR = SOURCE_PROJECT / "models"
SOURCE_DATA_DIR = SOURCE_PROJECT / "data"
SOURCE_PROCESSED_DATA_DIR = SOURCE_DATA_DIR / "processed"

SOURCE_REPORT_DIR = SOURCE_PROJECT / "reports"
SOURCE_FIGURE_DIR = SOURCE_REPORT_DIR / "figures"
SOURCE_TABLE_DIR = SOURCE_REPORT_DIR / "tables"


Project: Capstone3


## Load Capstone Two Assets

The cleaned analytical dataset, trained Random Forest model, fitted scaler, and ordered model feature names developed during Capstone Two are loaded below. These validated assets provide the foundation for the SHAP explainability and clustering analyses developed in Capstone Three.

In [4]:
# ---------------------------------------
# Load Capstone Two Assets
# ---------------------------------------

# Load the cleaned country-level analytical dataset
analysis_df = pd.read_csv(
    SOURCE_PROCESSED_DATA_DIR / "covid_analysis_dataset.csv"
)

print(
    f"Dataset loaded: {analysis_df.shape[0]:,} countries × "
    f"{analysis_df.shape[1]:,} variables"
)

# Load the trained Random Forest model
rf_model = joblib.load(
    SOURCE_MODEL_DIR / "random_forest_model.joblib"
)

print("Random Forest model loaded")

# Load the StandardScaler used during model development
scaler = joblib.load(
    SOURCE_MODEL_DIR / "standard_scaler.joblib"
)

print("StandardScaler loaded")

# Load the ordered feature names expected by the model
feature_names = joblib.load(
    SOURCE_MODEL_DIR / "feature_names.joblib"
)

print(f"{len(feature_names)} model feature names loaded")

Dataset loaded: 239 countries × 22 variables
Random Forest model loaded
StandardScaler loaded
12 model feature names loaded


## Asset Validation

Verify that the data and model assets loaded successfully and are consistent with one another before performing additional preprocessing.

In [5]:
# ---------------------------------------
# Asset Validation
# ---------------------------------------

print(f"Dataset:            {analysis_df.shape}")
print(f"Model type:         {type(rf_model).__name__}")
print(f"Scaler type:        {type(scaler).__name__}")
print(f"Model features:     {len(feature_names)}")

display(analysis_df.head())

Dataset:            (239, 22)
Model type:         RandomForestRegressor
Scaler type:        StandardScaler
Model features:     12


,country,code,continent,population,total_cases,total_cases_per_million,total_deaths,total_deaths_per_million,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,stringency_index,median_age,gdp_per_capita,hospital_beds_per_thousand,life_expectancy,diabetes_prevalence,iso_code,country_wb,population_density,health_expenditure_per_capita,health_expenditure_pct_gdp
0,Afghanistan,AFG,Asia,40578847.0,235214.0,5796.4683,7998.0,197.09776,47.195450,45.270844,6.727495,27.394580,16.752001,1983.812622,0.35,65.616997,11.7,AFG,Afghanistan,61.328691,81.521126,21.508444
1,Albania,ALB,Europe,2827614.0,337234.0,119264.5100,3608.0,1275.98740,47.717087,45.244260,14.230054,41.782108,35.943001,21641.074219,2.90,78.768799,10.6,ALB,Albania,90.867226,465.570435,7.357504
2,Algeria,DZA,Africa,45477391.0,272440.0,5990.6690,6881.0,151.30595,17.239624,14.251447,1.265796,48.367728,27.983000,15501.919922,1.61,76.128899,17.5,DZA,Algeria,18.793445,208.939117,5.021889
3,American Samoa,ASM,Oceania,48365.0,8359.0,172831.6000,34.0,702.98770,NaN,NaN,NaN,NaN,27.927000,NaN,NaN,72.752098,NaN,ASM,American Samoa,246.125000,NaN,NaN
4,Andorra,AND,Europe,79722.0,48015.0,602280.4400,159.0,1994.43070,72.643684,67.109460,54.026493,33.486770,42.832001,65928.304688,NaN,84.016403,10.1,AND,Andorra,166.731915,3668.447510,8.646717


In [6]:
# ---------------------------------------
# Verify Model Input Features
# ---------------------------------------

model_features = set(feature_names)
dataset_features = set(analysis_df.columns)

missing = model_features - dataset_features

print(f"Dataset variables: {len(dataset_features)}")
print(f"Model features:    {len(model_features)}")

if len(missing) == 0:
    print("✓ Dataset contains all required model features.")
else:
    print("\nAdditional preprocessing is required.")
    print("The following model features are created during preprocessing:")
    print(sorted(missing))

Dataset variables: 22
Model features:    12

Additional preprocessing is required.
The following model features are created during preprocessing:
['continent_Asia', 'continent_Europe', 'continent_North America', 'continent_Oceania', 'continent_South America']


### Interpretation

The analytical dataset stores the original categorical variable (`continent`) rather than the one-hot encoded variables used during model training. These derived variables will be recreated during preprocessing to ensure compatibility with the trained Random Forest model.

This validation confirms that the cleaned analytical dataset contains the information necessary to reconstruct all model inputs.

## Preparing Data for Explainable AI and Clustering

## Preparing Data for Explainable AI (SHAP)

SHAP (SHapley Additive exPlanations) provides local and global explanations for machine learning predictions by quantifying the contribution of each predictor to an individual prediction.

For tree-based models such as Random Forest, SHAP operates directly on the trained model and therefore requires very little additional preprocessing. However, the feature matrix supplied to SHAP must exactly match the feature representation used during model training, including feature order and one-hot encoded categorical variables.

This section verifies that the analytical dataset can be transformed into the model input required for SHAP analysis.

## Preparing the Model Feature Matrix

The Random Forest model developed in Capstone Two was trained using a processed feature matrix that included one-hot encoded continent variables. The analytical dataset stores the original `continent` variable, so the model input matrix must be reconstructed before performing SHAP analysis.

This section recreates the feature matrix expected by the trained model while preserving the original analytical dataset.

In [7]:
# ---------------------------------------
# Prepare Model Feature Matrix
# ---------------------------------------

# Create a working copy of the analytical dataset.
# The original analysis_df remains unchanged throughout the project.
model_df = analysis_df[
    [
        "median_age",
        "life_expectancy",
        "total_cases_per_million",
        "gdp_per_capita",
        "hospital_beds_per_thousand",
        "people_fully_vaccinated_per_hundred",
        "stringency_index",
        "continent",
    ]
].copy()

In [8]:
# ---------------------------------------
# Impute Missing Values
# ---------------------------------------

# The Random Forest model was developed using median imputation
# for numerical predictors. Recreate the same preprocessing
# before reconstructing the model feature matrix.

numeric_cols = [
    "median_age",
    "life_expectancy",
    "total_cases_per_million",
    "gdp_per_capita",
    "hospital_beds_per_thousand",
    "people_fully_vaccinated_per_hundred",
    "stringency_index",
]

imputer = SimpleImputer(strategy="median")

model_df[numeric_cols] = imputer.fit_transform(
    model_df[numeric_cols]
)

joblib.dump(
    imputer,
    MODEL_DIR / "median_imputer.joblib"
)

print("✓ Missing values imputed using median values.")

✓ Missing values imputed using median values.


In [9]:
# ---------------------------------------
# Create Dummy Variables
# ---------------------------------------

# Recreate the one-hot encoded continent variables used during
# Random Forest model training in Capstone Two.
model_df = pd.get_dummies(
    model_df,
    columns=["continent"],
    drop_first=True,
    dtype=float,
)

# Arrange the columns in the exact order expected by the trained model.
# Any missing dummy variables are automatically created and filled with zero.
model_df = model_df.reindex(
    columns=feature_names,
    fill_value=0.0,
)

# Display the dimensions of the reconstructed model feature matrix.
print(f"Model feature matrix: {model_df.shape}")

# Display the first few observations for verification.
display(model_df.head())

Model feature matrix: (239, 12)


,median_age,life_expectancy,total_cases_per_million,gdp_per_capita,hospital_beds_per_thousand,people_fully_vaccinated_per_hundred,stringency_index,continent_Asia,continent_Europe,continent_North America,continent_Oceania,continent_South America
0,16.752001,65.616997,5796.4683,1983.812622,0.35,45.270844,27.394580,1.0,0.0,0.0,0.0,0.0
1,35.943001,78.768799,119264.5100,21641.074219,2.90,45.244260,41.782108,0.0,1.0,0.0,0.0,0.0
2,27.983000,76.128899,5990.6690,15501.919922,1.61,14.251447,48.367728,0.0,0.0,0.0,0.0,0.0
3,27.927000,72.752098,172831.6000,18068.505859,2.38,62.440320,43.273038,0.0,0.0,0.0,1.0,0.0
4,42.832001,84.016403,602280.4400,65928.304688,2.38,67.109460,33.486770,0.0,1.0,0.0,0.0,0.0


In [10]:
# ---------------------------------------
# Validate Model Feature Matrix
# ---------------------------------------

# Confirm that the reconstructed feature matrix exactly matches
# the feature order expected by the trained Random Forest model.
assert list(model_df.columns) == list(feature_names), (
    "Model feature columns do not match the saved feature order."
)

print("✓ Model feature columns match the saved feature order.")
print(f"Number of model features: {model_df.shape[1]}")

# Identify variables that still require missing-value preprocessing.
missing_summary = (
    model_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0]

if missing_summary.empty:
    print("✓ No missing values remain in the model feature matrix.")
else:
    print("\nMissing values requiring preprocessing:")
    display(missing_summary.to_frame(name="Missing Values"))

✓ Model feature columns match the saved feature order.
Number of model features: 12
✓ No missing values remain in the model feature matrix.


In [11]:
# ---------------------------------------
# Compare Analytical and Model Datasets
# ---------------------------------------

print(f"Analytical dataset:    {analysis_df.shape}")
print(f"Model feature matrix:  {model_df.shape}")

print("\nAnalytical variables:")
print(list(analysis_df.columns))

print("\nModel variables:")
print(list(model_df.columns))

Analytical dataset:    (239, 22)
Model feature matrix:  (239, 12)

Analytical variables:
['country', 'code', 'continent', 'population', 'total_cases', 'total_cases_per_million', 'total_deaths', 'total_deaths_per_million', 'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred', 'total_boosters_per_hundred', 'stringency_index', 'median_age', 'gdp_per_capita', 'hospital_beds_per_thousand', 'life_expectancy', 'diabetes_prevalence', 'iso_code', 'country_wb', 'population_density', 'health_expenditure_per_capita', 'health_expenditure_pct_gdp']

Model variables:
['median_age', 'life_expectancy', 'total_cases_per_million', 'gdp_per_capita', 'hospital_beds_per_thousand', 'people_fully_vaccinated_per_hundred', 'stringency_index', 'continent_Asia', 'continent_Europe', 'continent_North America', 'continent_Oceania', 'continent_South America']


In [15]:
# ---------------------------------------
# Verify Model Compatibility
# ---------------------------------------

# Scale only the numeric predictors used to fit the saved StandardScaler.
numeric_features = list(scaler.feature_names_in_)

model_input_df = model_df.copy()

model_input_df[numeric_features] = scaler.transform(
    model_df[numeric_features]
)

# Preserve the exact feature order expected by the trained Random Forest.
model_input_df = model_input_df[feature_names]

# Confirm that the prepared matrix matches the model specification.
assert list(model_input_df.columns) == list(feature_names), (
    "Prepared model inputs do not match the saved feature order."
)

assert model_input_df.isna().sum().sum() == 0, (
    "Missing values remain in the prepared model inputs."
)

# Generate predictions to verify that all saved assets work together.
model_predictions = rf_model.predict(model_input_df)

assert len(model_predictions) == len(analysis_df), (
    "The number of predictions does not match the number of countries."
)

print("✓ Saved preprocessing assets and Random Forest model are compatible.")
print(f"Predictions generated for {len(model_predictions):,} countries.")

✓ Saved preprocessing assets and Random Forest model are compatible.
Predictions generated for 239 countries.


In [16]:
# ---------------------------------------
# Review Reconstructed Predictions
# ---------------------------------------

# Pair each country with its observed and reconstructed model prediction
# to confirm that the prepared inputs produce plausible results.
prediction_check_df = analysis_df[
    [
        "country",
        "continent",
        "total_deaths_per_million",
    ]
].copy()

prediction_check_df["predicted_deaths_per_million"] = model_predictions

prediction_check_df["prediction_error"] = (
    prediction_check_df["predicted_deaths_per_million"]
    - prediction_check_df["total_deaths_per_million"]
)

display(prediction_check_df.head())

,country,continent,total_deaths_per_million,predicted_deaths_per_million,prediction_error
0,Afghanistan,Asia,197.09776,106.520683,-90.577077
1,Albania,Europe,1275.98740,1602.220766,326.233366
2,Algeria,Africa,151.30595,303.154977,151.849027
3,American Samoa,Oceania,702.98770,761.575014,58.587314
4,Andorra,Europe,1994.43070,1903.709640,-90.721060


In [17]:
# Review prediction availability and observed-outcome missingness.
print(
    f"Predictions available: "
    f"{prediction_check_df['predicted_deaths_per_million'].notna().sum():,}"
)

print(
    f"Observed mortality values available: "
    f"{prediction_check_df['total_deaths_per_million'].notna().sum():,}"
)

Predictions available: 239
Observed mortality values available: 233


In [18]:
# ---------------------------------------
# Save Model Datasets
# ---------------------------------------

# Save the reconstructed model dataset (unscaled).
model_df.to_csv(
    PROCESSED_DATA_DIR / "covid_model_dataset.csv",
    index=False,
)

# Save the fully prepared model input matrix (scaled).
model_input_df.to_csv(
    PROCESSED_DATA_DIR / "covid_model_input_dataset.csv",
    index=False,
)

print("✓ Model datasets and preprocessing assets saved.")

✓ Model datasets and preprocessing assets saved.


In [19]:
print("Created during this notebook:")

print("  ✓ covid_model_dataset.csv")
print("  ✓ covid_model_input_dataset.csv")
print("  ✓ median_imputer.joblib")

Created during this notebook:
  ✓ covid_model_dataset.csv
  ✓ covid_model_input_dataset.csv
  ✓ median_imputer.joblib


## Preparing Data for Country Clustering

In [20]:
# ---------------------------------------
# Assemble Candidate Clustering Dataset
# ---------------------------------------

# Include plausible country descriptors that may help identify meaningful
# similarity patterns. Identifiers, continent, and mortality are retained
# for interpretation but will not be used directly to form clusters.
candidate_cluster_features = [
    "median_age",
    "life_expectancy",
    "gdp_per_capita",
    "hospital_beds_per_thousand",
    "health_expenditure_per_capita",
    "health_expenditure_pct_gdp",
    "people_fully_vaccinated_per_hundred",
    "total_boosters_per_hundred",
    "population_density",
    "diabetes_prevalence",
    "total_cases_per_million",
    "stringency_index",
]

cluster_candidate_df = analysis_df[
    [
        "country",
        "continent",
        "total_deaths_per_million",
        *candidate_cluster_features,
    ]
].copy()

print(f"Candidate clustering dataset: {cluster_candidate_df.shape}")
display(cluster_candidate_df.head())

Candidate clustering dataset: (239, 15)


,country,continent,total_deaths_per_million,median_age,life_expectancy,gdp_per_capita,hospital_beds_per_thousand,health_expenditure_per_capita,health_expenditure_pct_gdp,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,population_density,diabetes_prevalence,total_cases_per_million,stringency_index
0,Afghanistan,Asia,197.09776,16.752001,65.616997,1983.812622,0.35,81.521126,21.508444,45.270844,6.727495,61.328691,11.7,5796.4683,27.394580
1,Albania,Europe,1275.98740,35.943001,78.768799,21641.074219,2.90,465.570435,7.357504,45.244260,14.230054,90.867226,10.6,119264.5100,41.782108
2,Algeria,Africa,151.30595,27.983000,76.128899,15501.919922,1.61,208.939117,5.021889,14.251447,1.265796,18.793445,17.5,5990.6690,48.367728
3,American Samoa,Oceania,702.98770,27.927000,72.752098,NaN,NaN,NaN,NaN,NaN,NaN,246.125000,NaN,172831.6000,NaN
4,Andorra,Europe,1994.43070,42.832001,84.016403,65928.304688,NaN,3668.447510,8.646717,67.109460,54.026493,166.731915,10.1,602280.4400,33.486770


In [21]:
# ---------------------------------------
# Evaluate Candidate Clustering Data
# ---------------------------------------

# Summarize completeness and data types before deciding how each candidate
# feature should be treated in the clustering analysis.
cluster_quality_df = pd.DataFrame(
    {
        "Data Type": cluster_candidate_df[candidate_cluster_features].dtypes,
        "Missing Values": cluster_candidate_df[
            candidate_cluster_features
        ].isna().sum(),
        "Missing Percent": (
            cluster_candidate_df[candidate_cluster_features]
            .isna()
            .mean()
            .mul(100)
            .round(1)
        ),
        "Unique Values": cluster_candidate_df[
            candidate_cluster_features
        ].nunique(),
    }
).sort_values(
    "Missing Percent",
    ascending=False,
)

display(cluster_quality_df)

,Data Type,Missing Values,Missing Percent,Unique Values
hospital_beds_per_thousand,float64,71,29.7,149
stringency_index,float64,54,22.6,185
health_expenditure_per_capita,float64,47,19.7,192
health_expenditure_pct_gdp,float64,46,19.2,193
gdp_per_capita,float64,40,16.7,199
total_boosters_per_hundred,float64,31,13.0,202
diabetes_prevalence,float64,30,12.6,122
people_fully_vaccinated_per_hundred,float64,24,10.0,215
population_density,float64,24,10.0,215
total_cases_per_million,float64,6,2.5,232


In [22]:
# ---------------------------------------
# Check Country-Level Uniqueness
# ---------------------------------------

# Each row should represent one country, so duplicate country records would
# need to be resolved before clustering.
duplicate_countries = cluster_candidate_df["country"].duplicated().sum()
duplicate_rows = cluster_candidate_df.duplicated().sum()

print(f"Duplicate country names: {duplicate_countries}")
print(f"Fully duplicated rows:   {duplicate_rows}")

assert duplicate_countries == 0, (
    "Duplicate country records were found in the candidate clustering dataset."
)

Duplicate country names: 0
Fully duplicated rows:   0


### Observations

The candidate clustering dataset contains one observation for each of the 239 countries in the analytical dataset, and no duplicate country records were identified.

The candidate variables exhibited varying levels of missingness, ranging from less than 1% for **median age** and **life expectancy** to approximately 30% for **hospital beds per thousand**. Most candidate variables contained fewer than 20% missing values, suggesting they remain suitable for further investigation during exploratory data analysis.

At this stage, no candidate variables were excluded solely because of missing values. Instead, the purpose of this notebook is to assemble a comprehensive set of plausible clustering variables. The final feature selection will be guided by exploratory data analysis, including evaluation of variable distributions, correlations, redundancy, and interpretability, rather than by missingness alone.

The completed candidate clustering dataset will serve as the starting point for the exploratory analysis notebook, where the final clustering feature set and preprocessing pipeline will be determined.

In [23]:
# ---------------------------------------
# Save Candidate Clustering Dataset
# ---------------------------------------

# Save the candidate clustering dataset for use during
# exploratory data analysis.
cluster_candidate_df.to_csv(
    PROCESSED_DATA_DIR / "covid_cluster_candidate_dataset.csv",
    index=False,
)

print("✓ Candidate clustering dataset saved.")

✓ Candidate clustering dataset saved.


## Notebook Summary

This notebook prepared the data required for the new analytical components introduced in Capstone Three.

Major accomplishments include:

- Loaded and validated all reusable assets developed during Capstone Two.
- Reconstructed the complete Random Forest preprocessing pipeline, including median imputation, one-hot encoding, and feature scaling.
- Verified compatibility between the reconstructed feature matrix, the saved preprocessing assets, and the trained Random Forest model.
- Generated predictions for all countries using the reconstructed preprocessing pipeline.
- Created a candidate clustering dataset containing demographic, healthcare, economic, vaccination, policy, and COVID-related variables.
- Evaluated candidate clustering variables for completeness and data quality.
- Saved the reconstructed datasets and preprocessing assets for use in subsequent notebooks.

The next notebook will perform exploratory analysis of the candidate clustering variables, select the final clustering feature set, and develop SHAP explanations and country clusters.

In [24]:
# ---------------------------------------
# Assets Created
# ---------------------------------------

print("Processed datasets")
print("------------------")
print("✓ covid_model_dataset.csv")
print("✓ covid_model_input_dataset.csv")
print("✓ covid_cluster_candidate_dataset.csv")

print("\nModel assets")
print("------------")
print("✓ median_imputer.joblib")

Processed datasets
------------------
✓ covid_model_dataset.csv
✓ covid_model_input_dataset.csv
✓ covid_cluster_candidate_dataset.csv

Model assets
------------
✓ median_imputer.joblib
